# KARANAM CHAITRA | ENG24CS0093 | 5B-B1 | 3/11/2026
## EXP 7: Decision Tree

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

In [ ]:
df = pd.read_csv('./diabetes (1).csv')
df

In [ ]:
print("Dataset shape:", df.shape)

In [ ]:
print("\nColumns:", list(df.columns))

In [ ]:
df.describe()

In [ ]:
df.head()

In [ ]:
zero_as_missing_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
print("\nZero-value counts (treated as missing) before cleaning:")
print(df[zero_as_missing_cols].eq(0).sum())

In [ ]:
for col in zero_as_missing_cols:
    median_value = df.loc[df[col] != 0, col].median()
    df[col] = df[col].replace(0, median_value)

print("\nZero-value counts after cleaning:")
print(df[zero_as_missing_cols].eq(0).sum())

In [ ]:
X = df.drop("Outcome", axis=1)
y = df["Outcome"]
print("\nTarget distribution:\n", y.value_counts())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTraining samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")

In [ ]:
dt_model = DecisionTreeClassifier(
    criterion="gini",
    max_depth=4,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)
dt_model.fit(X_train, y_train)

In [ ]:
# STEP 7: Evaluate the baseline model
y_pred = dt_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\nTest Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=["No Diabetes", "Diabetes"]))
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

In [ ]:
# STEP 8: Cross-validation
cv_scores = cross_val_score(dt_model, X, y, cv=5)
print(
    f"\n5-Fold Cross-Validation Accuracy: {cv_scores.mean()*100:.2f}% "
    f" (+/- {cv_scores.std()*100:.2f}%)"
)

In [ ]:
# STEP 9: Hyperparameter tuning using GridSearchCV
param_grid = {
    "max_depth": [3, 4, 5, 6, 8, None],
    "criterion": ["gini", "entropy"],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 3, 5]
}
grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

In [ ]:
grid_search.fit(X_train, y_train)
print("\nBest parameters found:", grid_search.best_params_)
print(f"Best cross-validation accuracy: {grid_search.best_score_*100:.2f}%")
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)
print(f"Tuned model test accuracy: {accuracy_score(y_test, y_pred_best)*100:.2f}%")

In [ ]:
# STEP 10: Feature Importance
importances = pd.Series(best_model.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=False)
print("\nFeature Importances (tuned model):\n", importances)

In [ ]:
# STEP 11: Visualizations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Feature Importance
importances.plot(kind="barh", ax=axes[0], color="firebrick")
axes[0].invert_yaxis()
axes[0].set_title("Feature Importance - Diabetes Prediction")
axes[0].set_xlabel("Importance Score")

# Confusion Matrix
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No Diabetes", "Diabetes"]
)
disp.plot(ax=axes[1], cmap="Oranges", colorbar=False)
axes[1].set_title("Confusion Matrix (baseline model)")

# Layout and save
plt.tight_layout()
plt.savefig("diabetes_dt_results.png", dpi=150)
plt.show()
print("\nPlots saved as 'diabetes_dt_results.png'")

In [ ]:
# STEP 12: Visualize the actual decision tree structure
plt.figure(figsize=(20, 10))
plot_tree(
    dt_model,
    feature_names=X.columns,
    class_names=["No Diabetes", "Diabetes"],
    filled=True,
    rounded=True,
    fontsize=9
)
plt.title("Decision Tree Structure (max_depth=4)")
plt.savefig(
    "diabetes_tree_structure.png",
    dpi=150,
    bbox_inches="tight"
)
plt.show()
print("Tree diagram saved as 'diabetes_tree_structure.png'")

In [ ]:
# STEP 13: Predict for a new patient
new_patient = pd.DataFrame({
    "Pregnancies": [2],
    "Glucose": [150],
    "BloodPressure": [80],
    "SkinThickness": [30],
    "Insulin": [100],
    "BMI": [33.5],
    "DiabetesPedigreeFunction": [0.6],
    "Age": [45]
})

In [ ]:
# Make prediction
prediction = best_model.predict(new_patient)[0]
# Calculate prediction probabilities
probability = best_model.predict_proba(new_patient)[0]
print(
    f"\nNew patient prediction: "
    f"{'Diabetes' if prediction == 1 else 'No Diabetes'}"
)
print(f"Probability of diabetes: {probability[1]*100:.2f}%")